In [2]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpz9mb5t1r".


In [3]:
%%cuda
#include <iostream>

// THE KERNEL (2D)
__global__ void matrixAdd2D(float* A, float* B, float* C, int width) {
    // Mapping threads to 2D coordinates
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // Flattening 2D index to 1D memory address
    if (col < width && row < width) {
        int index = row * width + col;
        C[index] = A[index] + B[index];
    }
}

int main() {
    int width = 4; // 4x4 Matrix
    size_t size = width * width * sizeof(float);

    float *h_A = (float*)malloc(size), *h_B = (float*)malloc(size), *h_C = (float*)malloc(size);

    // Initialize: A = 1s, B = 2s
    for(int i=0; i < width*width; i++) { h_A[i] = 1.0f; h_B[i] = 2.0f; }

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size); cudaMalloc(&d_B, size); cudaMalloc(&d_C, size);
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Launch Config: 2D Blocks and 2D Grid
    dim3 threadsPerBlock(2, 2); // 2x2 threads per block
    dim3 blocksPerGrid(2, 2);   // 2x2 blocks per grid (Total 4x4 threads)

    matrixAdd2D<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, width);

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    std::cout << "Result Matrix (C):" << std::endl;
    for(int i=0; i<width; i++) {
        for(int j=0; j<width; j++) {
            std::cout << h_C[i * width + j] << " ";
        }
        std::cout << std::endl;
    }

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    return 0;
}

Result Matrix (C):
3 3 3 3 
3 3 3 3 
3 3 3 3 
3 3 3 3 

